# 🛡️ VisionGuard: Train on Google Colab (T4 GPU)

This notebook trains and evaluates **PatchCore** and **EfficientAD** on **MVTec AD** using Colab's free **T4 GPU**.

### Steps Overview:
1. Verify T4 GPU
2. Clone/Upload VisionGuard repository
3. Install dependencies
4. Download MVTec AD directly into Colab (~1 min)
5. Train **PatchCore** on `bottle` (~2–3 mins on T4)
6. Train **EfficientAD** on `bottle`
7. Run quantitative evaluation against ground-truth defect masks
8. Download trained `.pt` checkpoints back to your local PC

## 1. Verify T4 GPU Accelerator
Make sure your Colab Runtime is set to: **Runtime -> Change runtime type -> T4 GPU**

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Install Dependencies

In [ ]:
!pip install --upgrade pip
!pip install anomalib==2.7.0 "torch>=2.0.0" "torchvision>=0.15.0" opencv-python scikit-learn pandas matplotlib pyyaml

## 3. Upload or Clone VisionGuard Repository
If you have uploaded your `DL_Project` folder to Google Drive or GitHub, clone or copy it here. Alternatively, run the cell below to set up the VisionGuard directory structure directly:

In [ ]:
import os
# Create local folder structure if starting clean
os.makedirs("datasets/mvtec_ad", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("reports", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)
print("Directories initialized.")

## 4. Download MVTec AD Directly in Colab (Fast)
Colab cloud bandwidth downloads the dataset in under 1 minute.

In [ ]:
# Download the official MVTec AD archive directly
!wget -q --show-progress https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f282/download/420938113-1629952094/mvtec_anomaly_detection.tar.xz -O mvtec_ad.tar.xz

# Extract 'bottle' category to save time and disk space (or remove 'bottle' to extract everything)
!tar -xf mvtec_ad.tar.xz -C datasets/mvtec_ad/ bottle
!ls -la datasets/mvtec_ad/bottle

## 5. Train PatchCore on T4 GPU
Builds the normal feature memory bank with coreset sampling using WideResNet-50-2.

In [ ]:
import os
os.environ["PYTHONPATH"] = "src:."

!python scripts/train.py \
    --model patchcore \
    --dataset mvtec_ad \
    --category bottle \
    --root ./datasets/mvtec_ad \
    --checkpoints ./checkpoints

## 6. Train EfficientAD on T4 GPU

In [ ]:
!python scripts/train.py \
    --model efficientad \
    --dataset mvtec_ad \
    --category bottle \
    --root ./datasets/mvtec_ad \
    --checkpoints ./checkpoints

## 7. Evaluate Models Against Ground Truth Masks
Computes real image AUROC, pixel AUROC, F1, and IoU.

In [ ]:
# Evaluate PatchCore
!python scripts/evaluate.py --model patchcore --dataset mvtec_ad --category bottle

# Evaluate EfficientAD
!python scripts/evaluate.py --model efficientad --dataset mvtec_ad --category bottle

## 8. Download Checkpoints to Local PC
Run this cell to download the trained `.pt` files. Place them into your local `checkpoints/` folder to use them instantly with Gradio & Flask!

In [ ]:
from google.colab import files
import os

for f in os.listdir("checkpoints"):
    if f.endswith(".pt"):
        print(f"Downloading {f}...")
        files.download(os.path.join("checkpoints", f))